# Cartographie des Points Noirs — Réseau Ferroviaire Belge
 
**Source :** Open Data Infrabel  
**Stack :** Python · Pandas · Folium · Plotly  

---
## Plan du notebook
1. Installation des dépendances
2. Téléchargement des données Infrabel
3. Exploration des données brutes
4. Nettoyage et transformation
5. Agrégation par gare
6. Cartographie interactive (Folium)
7. Analyse graphique (Plotly)
8. Conclusions & insights

---
##  Étape 0 — Installation des dépendances

In [1]:
# Installation des librairies nécessaires
# À exécuter une seule fois
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'pandas', 'folium', 'plotly', 'requests', 'openpyxl',
                '--quiet'])
print('✅ Librairies installées')

✅ Librairies installées


---
## Imports

In [2]:
import requests
import pandas as pd
import folium
from folium.plugins import HeatMap
import plotly.express as px
import plotly.graph_objects as go
from io import StringIO
import os
import warnings
warnings.filterwarnings('ignore')

# Dossiers de sortie
os.makedirs('data/raw',   exist_ok=True)
os.makedirs('data/clean', exist_ok=True)

print('✅ Imports OK')

✅ Imports OK


---
## 1️ Téléchargement des données Infrabel

In [3]:
# --- Récupération de l'index des fichiers mensuels ---
INDEX_URL = (
    'https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets/'
    'stiptheid-gegevens-maandelijksebestanden/records?limit=100&lang=fr'
)

print('Récupération de l\'index...')
resp = requests.get(INDEX_URL, timeout=30)
df_index = pd.DataFrame(resp.json()['results'])
df_index = df_index.sort_values('mois', ascending=False).reset_index(drop=True)

print(f'    {len(df_index)} mois disponibles')
print(f'    Période : {df_index["mois"].iloc[-1]} → {df_index["mois"].iloc[0]}')
df_index.head()

Récupération de l'index...
    100 mois disponibles
    Période : 2014-03 → 2026-02


,mois,link_to_data
0,2026-02,https://fr.ftp.opendatasoft.com/infrabel/Punct...
1,2026-01,https://fr.ftp.opendatasoft.com/infrabel/Punct...
2,2025-12,https://fr.ftp.opendatasoft.com/infrabel/Punct...
3,2025-11,https://fr.ftp.opendatasoft.com/infrabel/Punct...
4,2025-10,https://fr.ftp.opendatasoft.com/infrabel/Punct...


In [4]:
# --- Téléchargement des 12 derniers mois ---
NB_MOIS = 12
df_recent = df_index.head(NB_MOIS)

print(f'🚄 Téléchargement des {NB_MOIS} derniers mois...')
print(f'   Période : {df_recent["mois"].iloc[-1]} → {df_recent["mois"].iloc[0]}\n')

all_dfs = []

for _, row in df_recent.iterrows():
    mois = row['mois']
    url  = row['link_to_data']
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        df_month = pd.read_csv(StringIO(r.text), sep=',', encoding='utf-8')
        df_month['mois'] = mois
        all_dfs.append(df_month)
        print(f'   ✅ {mois} — {len(df_month):,} lignes | colonnes : {list(df_month.columns)[:4]}...')
    except Exception as e:
        print(f'   ⚠️  {mois} — Erreur : {e}')

# Fusion de tous les mois
df_raw = pd.concat(all_dfs, ignore_index=True)

# Sauvegarde brute
df_raw.to_csv('../data/raw/ponctualite_merged.csv', index=False, encoding='utf-8-sig')

print(f'\n Total : {len(df_raw):,} lignes | {len(df_raw.columns)} colonnes')
print(f'Sauvegardé : data/raw/ponctualite_merged.csv')

🚄 Téléchargement des 12 derniers mois...
   Période : 2025-01 → 2026-02

   ✅ 2026-02 — 1,897,533 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2026-01 — 1,889,339 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-12 — 2,066,672 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-11 — 1,757,601 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-10 — 2,060,066 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-09 — 2,015,606 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-08 — 1,861,871 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-07 — 1,916,850 lignes | colonnes : ['DATDEP', 'TRAIN_NO', 'RELATION', 'TRAIN_SERV']...
   ✅ 2025-05 — 1,919,566 lignes | colonnes : ['DATDEP', 'TRAIN_NO', 'RELATION', 'TRAIN_SERV']...
   ✅ 2025-04 — 1,770,587 lignes | colonnes : ['DATDEP', 'TRAIN_NO', 

---
## 2️ Exploration des données brutes

In [5]:
# Structure générale
print(f'Dimensions : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes')
print(f'\nColonnes :')
for col in df_raw.columns:
    print(f'  - {col} ({df_raw[col].dtype}) | ex: {df_raw[col].iloc[0]}')

Dimensions : 22,642,916 lignes × 23 colonnes

Colonnes :
  - DATDEP (object) | ex: 01FEB2026
  - CIRC_TYP (int64) | ex: 1
  - TRAIN_NO (int64) | ex: 2623
  - RELATION (object) | ex: IC 08
  - TRAIN_SERV (object) | ex: SNCB/NMBS
  - OP1_COD (object) | ex: nan
  - THOP1_COD (object) | ex: nan
  - PTCAR_NO (int64) | ex: 810
  - PTCAR_LG_NM_NL (object) | ex: MECHELEN
  - LINE_NO_DEP (object) | ex: 25
  - REAL_DATE_ARR (object) | ex: nan
  - REAL_TIME_ARR (object) | ex: nan
  - REAL_DATE_DEP (object) | ex: 01FEB2026
  - REAL_TIME_DEP (object) | ex: 0:33:34
  - PLANNED_DATE_ARR (object) | ex: nan
  - PLANNED_TIME_ARR (object) | ex: nan
  - PLANNED_TIME_DEP (object) | ex: 0:33:00
  - PLANNED_DATE_DEP (object) | ex: 01FEB2026
  - DELAY_ARR (float64) | ex: nan
  - DELAY_DEP (float64) | ex: 34.0
  - RELATION_DIRECTION (object) | ex: IC 08: ANTWERPEN-CENTRAAL -> HASSELT
  - LINE_NO_ARR (object) | ex: nan
  - mois (object) | ex: 2026-02


In [6]:
display(df_raw.head())
df_raw.info()

,DATDEP,CIRC_TYP,TRAIN_NO,RELATION,TRAIN_SERV,OP1_COD,THOP1_COD,PTCAR_NO,PTCAR_LG_NM_NL,LINE_NO_DEP,...,REAL_TIME_DEP,PLANNED_DATE_ARR,PLANNED_TIME_ARR,PLANNED_TIME_DEP,PLANNED_DATE_DEP,DELAY_ARR,DELAY_DEP,RELATION_DIRECTION,LINE_NO_ARR,mois
0,01FEB2026,1,2623,IC 08,SNCB/NMBS,NaN,NaN,810,MECHELEN,25,...,0:33:34,NaN,NaN,0:33:00,01FEB2026,NaN,34.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,NaN,2026-02
1,01FEB2026,1,2623,IC 08,SNCB/NMBS,=,=,219,BRUSSELS AIRPORT - ZAVENTEM,36C,...,0:46:38,01FEB2026,0:44:00,0:47:00,01FEB2026,29.0,-22.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36C,2026-02
2,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,916,NOSSEGEM,36,...,0:50:14,01FEB2026,0:50:00,0:50:00,01FEB2026,14.0,14.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
3,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,648,KORTENBERG,36,...,0:51:40,01FEB2026,0:52:00,0:52:00,01FEB2026,-20.0,-20.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
4,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,33,KORTENBERG-GOEDEREN,36,...,0:51:47,01FEB2026,0:52:00,0:52:00,01FEB2026,-13.0,-13.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22642916 entries, 0 to 22642915
Data columns (total 23 columns):
 #   Column              Dtype  
---  ------              -----  
 0   DATDEP              object 
 1   CIRC_TYP            int64  
 2   TRAIN_NO            int64  
 3   RELATION            object 
 4   TRAIN_SERV          object 
 5   OP1_COD             object 
 6   THOP1_COD           object 
 7   PTCAR_NO            int64  
 8   PTCAR_LG_NM_NL      object 
 9   LINE_NO_DEP         object 
 10  REAL_DATE_ARR       object 
 11  REAL_TIME_ARR       object 
 12  REAL_DATE_DEP       object 
 13  REAL_TIME_DEP       object 
 14  PLANNED_DATE_ARR    object 
 15  PLANNED_TIME_ARR    object 
 16  PLANNED_TIME_DEP    object 
 17  PLANNED_DATE_DEP    object 
 18  DELAY_ARR           float64
 19  DELAY_DEP           float64
 20  RELATION_DIRECTION  object 
 21  LINE_NO_ARR         object 
 22  mois                object 
dtypes: float64(2), int64(3), object(18)
memory usage: 3.9+ 

In [7]:
# Valeurs manquantes

missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
pd.DataFrame({'Manquants': missing, 'Pct (%)': missing_pct})

,Manquants,Pct (%)
DATDEP,0,0.0
CIRC_TYP,0,0.0
TRAIN_NO,0,0.0
RELATION,0,0.0
TRAIN_SERV,0,0.0
OP1_COD,12086986,53.4
THOP1_COD,2243652,9.9
PTCAR_NO,0,0.0
PTCAR_LG_NM_NL,0,0.0
LINE_NO_DEP,1106960,4.9


In [8]:
# Aperçu des données
df_raw.head(10)

,DATDEP,CIRC_TYP,TRAIN_NO,RELATION,TRAIN_SERV,OP1_COD,THOP1_COD,PTCAR_NO,PTCAR_LG_NM_NL,LINE_NO_DEP,...,REAL_TIME_DEP,PLANNED_DATE_ARR,PLANNED_TIME_ARR,PLANNED_TIME_DEP,PLANNED_DATE_DEP,DELAY_ARR,DELAY_DEP,RELATION_DIRECTION,LINE_NO_ARR,mois
0,01FEB2026,1,2623,IC 08,SNCB/NMBS,NaN,NaN,810,MECHELEN,25,...,0:33:34,NaN,NaN,0:33:00,01FEB2026,NaN,34.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,NaN,2026-02
1,01FEB2026,1,2623,IC 08,SNCB/NMBS,=,=,219,BRUSSELS AIRPORT - ZAVENTEM,36C,...,0:46:38,01FEB2026,0:44:00,0:47:00,01FEB2026,29.0,-22.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36C,2026-02
2,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,916,NOSSEGEM,36,...,0:50:14,01FEB2026,0:50:00,0:50:00,01FEB2026,14.0,14.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
3,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,648,KORTENBERG,36,...,0:51:40,01FEB2026,0:52:00,0:52:00,01FEB2026,-20.0,-20.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
4,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,33,KORTENBERG-GOEDEREN,36,...,0:51:47,01FEB2026,0:52:00,0:52:00,01FEB2026,-13.0,-13.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
5,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,368,ERPS-KWERPS,36,...,0:52:55,01FEB2026,0:53:00,0:53:00,01FEB2026,-5.0,-5.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
6,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,1174,VELTEM,36,...,0:54:24,01FEB2026,0:55:00,0:55:00,01FEB2026,-36.0,-36.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
7,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,553,HERENT,36,...,0:55:23,01FEB2026,0:56:00,0:56:00,01FEB2026,-37.0,-37.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
8,01FEB2026,1,2623,IC 08,SNCB/NMBS,NaN,NaN,715,LEUVEN,NaN,...,NaN,01FEB2026,1:00:00,NaN,NaN,-42.0,NaN,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
9,01FEB2026,1,3229,IC 26,SNCB/NMBS,NaN,NaN,319,DENDERMONDE,60,...,9:05:12,NaN,NaN,9:05:00,01FEB2026,NaN,12.0,IC 26: DENDERMONDE -> BRUSSEL-ZUID,NaN,2026-02


In [9]:
df_raw['CIRC_TYP'].unique()

array([1])

In [10]:
df_raw.drop(columns=['CIRC_TYP'], inplace=True)

In [11]:
df_raw['OP1_COD'].unique()

array([nan, '=', 'D', 'P', '>', 'V', 'C', ')', '(', '<', ':'],
      dtype=object)

In [12]:
df_raw['THOP1_COD'].unique()

array([nan, '=', 'D', 'P', '>', 'V', 'C', ')', '(', '<'], dtype=object)

In [13]:
# Comparaison directe
(df_raw['OP1_COD'] == df_raw['THOP1_COD']).value_counts(dropna=False)

False    13880100
True      8762816
Name: count, dtype: int64

In [14]:
# Voir quelques lignes où ils diffèrent, avec le contexte
df_raw[df_raw['OP1_COD'] != df_raw['THOP1_COD']][
    ['RELATION', 'PTCAR_LG_NM_NL', 'OP1_COD', 'THOP1_COD', 'DELAY_DEP']
].head(20)

,RELATION,PTCAR_LG_NM_NL,OP1_COD,THOP1_COD,DELAY_DEP
0,IC 08,MECHELEN,NaN,NaN,34.0
8,IC 08,LEUVEN,NaN,NaN,NaN
9,IC 26,DENDERMONDE,NaN,NaN,12.0
24,IC 26,BRUSSEL-ZUID,NaN,NaN,NaN
25,IC 26,JETTE,NaN,=,1469.0
26,IC 26,BOCKSTAEL,P,D,1466.0
28,IC 26,BRUSSEL-CONGRES,P,D,1610.0
30,IC 26,BRUSSEL-KAPELLEKERK,P,D,1562.0
31,IC 26,BRUSSEL-ZUID,NaN,NaN,NaN
32,IC 06-2,BRUSSELS AIRPORT - ZAVENTEM,NaN,NaN,22.0


In [15]:
# convertir les veleurs en minutes
(df_raw['DELAY_DEP'] / 60).describe()


count    2.153612e+07
mean     2.009589e+00
std      5.661034e+00
min     -1.424700e+03
25%      1.166667e-01
50%      6.333333e-01
75%      2.016667e+00
max      6.446167e+02
Name: DELAY_DEP, dtype: float64

In [16]:
# Nettoyer les outliers et garder que les lignes avec un retard valide
df_valid = df_raw[
    (df_raw['DELAY_DEP'].notna()) &
    (df_raw['DELAY_DEP'] >= -30) &      # max 30min en avance (généreux)
    (df_raw['DELAY_DEP'] <= 1800)        # max 6h de retard
].copy()

df_valid['delay_min'] = df_valid['DELAY_DEP'] / 60

print(f'Lignes gardées : {len(df_valid):,} / {len(df_raw):,}')
display(df_valid['delay_min'].describe())

df_valid['delay_min'] = df_valid['DELAY_DEP'] / 60


Lignes gardées : 19,730,044 / 22,642,916


count    1.973004e+07
mean     1.976971e+00
std      3.545309e+00
min     -5.000000e-01
25%      2.000000e-01
50%      7.500000e-01
75%      2.166667e+00
max      3.000000e+01
Name: delay_min, dtype: float64

---
## 3️ Nettoyage et transformation

In [17]:
df = df_raw.copy()

# --- Affichage des colonnes disponibles pour adapter le nettoyage ---
print('Colonnes disponibles :', list(df.columns))

# --- Normalisation des noms de colonnes (minuscules, sans espaces) ---
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('(', '')
    .str.replace(')', '')
)
print('\nColonnes normalisées :', list(df.columns))

Colonnes disponibles : ['DATDEP', 'TRAIN_NO', 'RELATION', 'TRAIN_SERV', 'OP1_COD', 'THOP1_COD', 'PTCAR_NO', 'PTCAR_LG_NM_NL', 'LINE_NO_DEP', 'REAL_DATE_ARR', 'REAL_TIME_ARR', 'REAL_DATE_DEP', 'REAL_TIME_DEP', 'PLANNED_DATE_ARR', 'PLANNED_TIME_ARR', 'PLANNED_TIME_DEP', 'PLANNED_DATE_DEP', 'DELAY_ARR', 'DELAY_DEP', 'RELATION_DIRECTION', 'LINE_NO_ARR', 'mois']

Colonnes normalisées : ['datdep', 'train_no', 'relation', 'train_serv', 'op1_cod', 'thop1_cod', 'ptcar_no', 'ptcar_lg_nm_nl', 'line_no_dep', 'real_date_arr', 'real_time_arr', 'real_date_dep', 'real_time_dep', 'planned_date_arr', 'planned_time_arr', 'planned_time_dep', 'planned_date_dep', 'delay_arr', 'delay_dep', 'relation_direction', 'line_no_arr', 'mois']


In [18]:
df.rename(columns={
    'ptcar_lg_nm_nl': 'gare',
    'delay_dep': 'retard',
    'mois': 'date'
}, inplace=True)

df.head()  # → maintenant les colonnes sont 'gare', 'retard', 'date'

,datdep,train_no,relation,train_serv,op1_cod,thop1_cod,ptcar_no,gare,line_no_dep,real_date_arr,...,real_time_dep,planned_date_arr,planned_time_arr,planned_time_dep,planned_date_dep,delay_arr,retard,relation_direction,line_no_arr,date
0,01FEB2026,2623,IC 08,SNCB/NMBS,NaN,NaN,810,MECHELEN,25,NaN,...,0:33:34,NaN,NaN,0:33:00,01FEB2026,NaN,34.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,NaN,2026-02
1,01FEB2026,2623,IC 08,SNCB/NMBS,=,=,219,BRUSSELS AIRPORT - ZAVENTEM,36C,01FEB2026,...,0:46:38,01FEB2026,0:44:00,0:47:00,01FEB2026,29.0,-22.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36C,2026-02
2,01FEB2026,2623,IC 08,SNCB/NMBS,D,D,916,NOSSEGEM,36,01FEB2026,...,0:50:14,01FEB2026,0:50:00,0:50:00,01FEB2026,14.0,14.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
3,01FEB2026,2623,IC 08,SNCB/NMBS,D,D,648,KORTENBERG,36,01FEB2026,...,0:51:40,01FEB2026,0:52:00,0:52:00,01FEB2026,-20.0,-20.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
4,01FEB2026,2623,IC 08,SNCB/NMBS,D,D,33,KORTENBERG-GOEDEREN,36,01FEB2026,...,0:51:47,01FEB2026,0:52:00,0:52:00,01FEB2026,-13.0,-13.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02


In [19]:
# Colonnes de date à convertir en datetime
date_cols = ['datdep', 'real_date_arr', 'real_date_dep', 'planned_date_arr', 'planned_date_dep', 'date']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%d%b%Y', errors='coerce')

print('✅ Dates converties')
df[date_cols].dtypes


✅ Dates converties


datdep              datetime64[ns]
real_date_arr       datetime64[ns]
real_date_dep       datetime64[ns]
planned_date_arr    datetime64[ns]
planned_date_dep    datetime64[ns]
date                datetime64[ns]
dtype: object

In [20]:
df[date_cols]

,datdep,real_date_arr,real_date_dep,planned_date_arr,planned_date_dep,date
0,2026-02-01,NaT,2026-02-01,NaT,2026-02-01,NaT
1,2026-02-01,2026-02-01,2026-02-01,2026-02-01,2026-02-01,NaT
2,2026-02-01,2026-02-01,2026-02-01,2026-02-01,2026-02-01,NaT
3,2026-02-01,2026-02-01,2026-02-01,2026-02-01,2026-02-01,NaT
4,2026-02-01,2026-02-01,2026-02-01,2026-02-01,2026-02-01,NaT
...,...,...,...,...,...,...
22642911,2025-01-31,2025-01-31,2025-01-31,2025-01-31,2025-01-31,NaT
22642912,2025-01-31,NaT,2025-01-31,NaT,2025-01-31,NaT
22642913,2025-01-31,NaT,2025-02-01,NaT,2025-02-01,NaT
22642914,2025-01-31,2025-02-01,2025-02-01,2025-02-01,2025-02-01,NaT


In [21]:
# 1. Doublons
avant = len(df)
df = df.drop_duplicates()
print(f'Doublons supprimés : {avant - len(df):,}')

# 2. Nettoyer noms de gares
df['gare'] = df['gare'].astype(str).str.strip().str.title()
df = df[df['gare'] != 'Nan']

print(f'✅ {len(df):,} lignes propres')

Doublons supprimés : 0
✅ 22,642,916 lignes propres


In [22]:
print(df.columns.tolist())

['datdep', 'train_no', 'relation', 'train_serv', 'op1_cod', 'thop1_cod', 'ptcar_no', 'gare', 'line_no_dep', 'real_date_arr', 'real_time_arr', 'real_date_dep', 'real_time_dep', 'planned_date_arr', 'planned_time_arr', 'planned_time_dep', 'planned_date_dep', 'delay_arr', 'retard', 'relation_direction', 'line_no_arr', 'date']


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22642916 entries, 0 to 22642915
Data columns (total 22 columns):
 #   Column              Dtype         
---  ------              -----         
 0   datdep              datetime64[ns]
 1   train_no            int64         
 2   relation            object        
 3   train_serv          object        
 4   op1_cod             object        
 5   thop1_cod           object        
 6   ptcar_no            int64         
 7   gare                object        
 8   line_no_dep         object        
 9   real_date_arr       datetime64[ns]
 10  real_time_arr       object        
 11  real_date_dep       datetime64[ns]
 12  real_time_dep       object        
 13  planned_date_arr    datetime64[ns]
 14  planned_time_arr    object        
 15  planned_time_dep    object        
 16  planned_date_dep    datetime64[ns]
 17  delay_arr           float64       
 18  retard              float64       
 19  relation_direction  object        
 20  

In [24]:
cols_to_drop = [
    'train_no', 'train_serv', 'op1_cod', 'thop1_cod',
    'real_date_arr', 'real_date_dep', 'planned_date_arr', 'planned_date_dep',
    'real_time_arr', 'real_time_dep', 'planned_time_arr', 'planned_time_dep',
    'line_no_arr', 'relation_direction'
]

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(f'Colonnes restantes : {df.shape[1]} → {list(df.columns)}')
print(f'Mémoire : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Colonnes restantes : 8 → ['datdep', 'relation', 'ptcar_no', 'gare', 'line_no_dep', 'delay_arr', 'retard', 'date']
Mémoire : 5.15 GB


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22642916 entries, 0 to 22642915
Data columns (total 8 columns):
 #   Column       Dtype         
---  ------       -----         
 0   datdep       datetime64[ns]
 1   relation     object        
 2   ptcar_no     int64         
 3   gare         object        
 4   line_no_dep  object        
 5   delay_arr    float64       
 6   retard       float64       
 7   date         datetime64[ns]
dtypes: datetime64[ns](2), float64(2), int64(1), object(3)
memory usage: 1.3+ GB


In [26]:
df.head(10)

,datdep,relation,ptcar_no,gare,line_no_dep,delay_arr,retard,date
0,2026-02-01,IC 08,810,Mechelen,25,NaN,34.0,NaT
1,2026-02-01,IC 08,219,Brussels Airport - Zaventem,36C,29.0,-22.0,NaT
2,2026-02-01,IC 08,916,Nossegem,36,14.0,14.0,NaT
3,2026-02-01,IC 08,648,Kortenberg,36,-20.0,-20.0,NaT
4,2026-02-01,IC 08,33,Kortenberg-Goederen,36,-13.0,-13.0,NaT
5,2026-02-01,IC 08,368,Erps-Kwerps,36,-5.0,-5.0,NaT
6,2026-02-01,IC 08,1174,Veltem,36,-36.0,-36.0,NaT
7,2026-02-01,IC 08,553,Herent,36,-37.0,-37.0,NaT
8,2026-02-01,IC 08,715,Leuven,NaN,-42.0,NaN,NaT
9,2026-02-01,IC 26,319,Dendermonde,60,NaN,12.0,NaT


In [27]:
# optimise les types
for col in ['relation', 'gare', 'line_no_dep']:
    df[col] = df[col].astype('category')

print(f'Mémoire après catégorisation : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Mémoire après catégorisation : 1.04 GB


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22642916 entries, 0 to 22642915
Data columns (total 8 columns):
 #   Column       Dtype         
---  ------       -----         
 0   datdep       datetime64[ns]
 1   relation     category      
 2   ptcar_no     int64         
 3   gare         category      
 4   line_no_dep  category      
 5   delay_arr    float64       
 6   retard       float64       
 7   date         datetime64[ns]
dtypes: category(3), datetime64[ns](2), float64(2), int64(1)
memory usage: 993.4 MB


In [29]:
# Filtre outliers
df = df[
    (df['retard'].notna()) &
    (df['retard'] >= -30) &
    (df['retard'] <= 1800)
].copy()

# Retard en minutes
df['retard_min'] = (df['retard'] / 60).round(2)

print(f'Lignes finales : {len(df):,}')
print(f'Mémoire finale : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Lignes finales : 19,730,044
Mémoire finale : 1.22 GB


In [30]:

# Sauvegarde
import os
os.makedirs('data/clean', exist_ok=True)

df.to_parquet('data/clean/ponctualite_clean.parquet', index=False)

taille = os.path.getsize('data/clean/ponctualite_clean.parquet') / 1e6
print(f'Lignes sauvegardées : {len(df):,}')
print(f'Taille sur disque   : {taille:.1f} MB')
print(f'Colonnes            : {list(df.columns)}')

Lignes sauvegardées : 19,730,044
Taille sur disque   : 116.0 MB
Colonnes            : ['datdep', 'relation', 'ptcar_no', 'gare', 'line_no_dep', 'delay_arr', 'retard', 'date', 'retard_min']


---
## 4️ KPI's blackspots

In [31]:
# Agrégation par gare — création du DataFrame blackspots
blackspots = df.groupby('gare').agg(
    passages        = ('retard_min', 'count'),
    retard_moyen    = ('retard_min', 'mean'),
    pct_en_retard   = ('retard_min', lambda x: (x > 5).mean() * 100),
    minutes_perdues = ('retard_min', lambda x: x[x > 0].sum())
).round(2).reset_index()

print(f'Gares identifiées : {len(blackspots)}')
display(blackspots.head(10))

Gares identifiées : 673


,gare,passages,retard_moyen,pct_en_retard,minutes_perdues
0,Aalst,38204,1.59,7.72,60985.04
1,Aalst-Kerrebroek,2455,1.04,2.93,2683.65
2,Aalter,48258,1.70,8.34,82905.99
3,Aarschot,54198,1.86,9.44,101843.80
4,Aarsele,13919,2.24,10.76,31291.31
5,Acren,12510,2.60,14.00,32519.37
6,Aiseau,35460,1.64,7.99,59239.77
7,Alken,18788,2.84,14.97,53391.14
8,Amay,29534,1.93,10.12,57618.88
9,Ampsin,30102,2.03,10.27,61604.52


In [32]:
# Garder uniquement les gares avec assez de passages (vraies gares voyageurs)
blackspots = blackspots[blackspots['passages'] >= 5000].copy()
blackspots = blackspots.sort_values('minutes_perdues', ascending=False).reset_index(drop=True)

print(f'Gares significatives : {len(blackspots)}')
display(blackspots.head(15))

Gares significatives : 593


,gare,passages,retard_moyen,pct_en_retard,minutes_perdues
0,Brussel-Centraal,333254,2.94,16.25,978744.68
1,Brussel-Kapellekerk,311801,2.54,13.54,799178.43
2,Brussel-Congres,325361,2.43,13.71,798692.94
3,Brussel-Noord,327931,2.19,12.19,720146.27
4,Brussel-Zuid,306483,2.29,12.12,703366.10
5,Schaarbeek,213642,2.35,13.67,511062.92
6,Antwerpen-Berchem,220470,2.17,11.44,480534.92
7,Halle,122260,2.72,15.25,333606.49
8,Mortsel,137542,2.38,12.57,331181.86
9,Mechelen,143622,2.23,11.89,321706.30


---
## 5️ Coordonnées GPS des gares

In [33]:
import requests

#  1. Récupérer le référentiel iRail ─────────────────────────
url = "https://api.irail.be/stations/?format=json&lang=nl"
stations_ref = pd.DataFrame([{
    'gare_key' : s['name'].strip().upper(),
    'lat'      : float(s['locationY']),
    'lon'      : float(s['locationX'])
} for s in requests.get(url).json()['station']])

#  2. Clé de jointure sur blackspots ─────────────────────────
blackspots['gare_key'] = blackspots['gare'].str.upper().str.strip()

#  3. Toutes les corrections FR → NL ─────────────────────────
corrections = {
    'LIEGE-GUILLEMINS'    : 'LUIK-GUILLEMINS',
    'LIEGE-CARRE'         : 'LUIK-CARRE',
    'LIEGE-SAINT-LAMBERT' : 'LUIK-SINT-LAMBERTUS',
    'NAMUR'               : 'NAMEN',
    'CHARLEROI-CENTRAL'   : 'CHARLEROI-CENTRAAL',
    'MONS'                : 'BERGEN',
    'TUBIZE'              : 'TUBEKE',
    "BRAINE-L'ALLEUD"     : 'EIGENBRAKEL',
    'BRAINE-LE-COMTE'     : "'S GRAVENBRAKEL",
    'JURBISE'             : 'JURBEKE',
    'LA HULPE'            : 'TERHULPEN',
    'LOKEREN-OOST'        : 'LOKEREN',
    'SINT-NIKLAAS-OOST'   : 'SINT-NIKLAAS',
    'NIVELLES'            : 'NIJVEL',
    'BEVEREN(WAAS)'       : 'BEVEREN',
}
blackspots['gare_key'] = blackspots['gare_key'].replace(corrections)

#  4. Jointure ────────────────────────────────────────────────
bs_geo = blackspots.merge(stations_ref, on='gare_key', how='left')

#  5. Résultats ───────────────────────────────────────────────
matched   = bs_geo['lat'].notna().sum()
unmatched = bs_geo['lat'].isna().sum()
print(f'Avec coordonnées : {matched} / {len(bs_geo)}')
print(f'Sans coordonnées : {unmatched}')

if unmatched > 0:
    reste = bs_geo[bs_geo['lat'].isna()][['gare','passages']]\
            .sort_values('passages', ascending=False)
    print(f'\nNon-matchés (gares fret/techniques, à ignorer) :')
    print(reste.head(10))

Avec coordonnées : 500 / 593
Sans coordonnées : 93

Non-matchés (gares fret/techniques, à ignorer) :
                    gare  passages
49        Namur-Herbatte     81175
79       Braine-Alliance     50544
167      La Louviere-Sud     49450
107  Kortenberg-Goederen     46767
66            Hennuyeres     44402
161  Schaarbeek-Josaphat     42675
87     Hennuyeres-Garage     40911
97           Rhisnes-Sas     40164
52                Chenee     37084
94                Lonzee     37063


In [34]:
for terme in ['GRAVENBRAKEL', 'ALLIANCE', 'LOUVIERE', 'HERBAT', 
              'JURBEKE', 'TERHULPEN', 'HENNE', 'LOKEREN', 'HULPE']:
    matches = stations_ref[stations_ref['gare_key'].str.contains(terme, na=False)]
    if len(matches) > 0:
        print(f"{terme} → {matches['gare_key'].tolist()}")
    else:
        print(f"{terme} → ❌ absent d'iRail")

GRAVENBRAKEL → ["'S GRAVENBRAKEL"]
ALLIANCE → ❌ absent d'iRail
LOUVIERE → ❌ absent d'iRail
HERBAT → ❌ absent d'iRail
JURBEKE → ['JURBEKE']
TERHULPEN → ['TERHULPEN']
HENNE → ❌ absent d'iRail
LOKEREN → ['LOKEREN']
HULPE → ['TERHULPEN']


---
## 6️ Cartographie interactive (Folium)

In [35]:
import folium
from folium.plugins import HeatMap
import os

os.makedirs('outputs', exist_ok=True)

#  Rang sur bs_geo ───────────────────────────────────────────
bs_geo = bs_geo.sort_values('minutes_perdues', ascending=False).reset_index(drop=True)
bs_geo['rang'] = range(1, len(bs_geo) + 1)

#  Données avec coordonnées ──────────────────────────────────
df_map = bs_geo[bs_geo['lat'].notna()].copy()
df_map['intensite'] = df_map['minutes_perdues'] / df_map['minutes_perdues'].max()
heat_data = df_map[['lat', 'lon', 'intensite']].values.tolist()

#  Carte ─────────────────────────────────────────────────────
carte = folium.Map(location=[50.5, 4.5], zoom_start=9, tiles='CartoDB positron')

#  Titre ─────────────────────────────────────────────────────
titre = '''
<div style="position:fixed; top:15px; left:60px; z-index:1000;
     background:white; padding:10px 15px; border-radius:6px;
     border-left:4px solid #bd0026; font-family:Arial;
     box-shadow:2px 2px 6px rgba(0,0,0,0.2)">
    <b style="font-size:14px; color:#1a2940">Blackspots Infrabel</b><br>
    <span style="font-size:11px; color:#555">Minutes perdues par gare · 2025-2026</span>
</div>'''
carte.get_root().html.add_child(folium.Element(titre))

#  HeatMap ───────────────────────────────────────────────────
HeatMap(
    heat_data,
    radius=18,
    blur=15,
    max_zoom=10,
    gradient={0.2: '#ffffb2', 0.5: '#fd8d3c', 0.8: '#f03b20', 1.0: '#bd0026'}
).add_to(carte)

#  Marqueurs Top 15 ──────────────────────────────────────────
top15 = df_map.nlargest(15, 'minutes_perdues')

for _, row in top15.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color='#1a2940',
        fill=True,
        fill_color='#e74c3c',
        fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>{row['gare']}</b><br>"
            f"Rang : #{int(row['rang'])}<br>"
            f"Minutes perdues : {row['minutes_perdues']:,.0f}<br>"
            f"Retard moyen : {row['retard_moyen']:.1f} min<br>"
            f"% en retard : {row['pct_en_retard']:.1f}%",
            max_width=220
        ),
        tooltip=f"#{int(row['rang'])} {row['gare']}"
    ).add_to(carte)

#   Sauvegarde ────────────────────────────────────────────────
carte.save('outputs/blackspots_heatmap.html')
print('Carte sauvegardée : outputs/blackspots_heatmap.html')
carte

Carte sauvegardée : outputs/blackspots_heatmap.html


---
## 7️ Analyse graphique (Plotly)

In [36]:
# Analyse graphique (Plotly)

# Reconstruire la tendance depuis datdep (fiable) plutôt que date
tendance = df.copy()
tendance['mois'] = tendance['datdep'].dt.to_period('M').astype(str)

tendance = tendance.groupby('mois').agg(
    retard_moyen = ('retard_min', 'mean'),
    pct_en_retard = ('retard_min', lambda x: (x > 5).mean() * 100),
    total_passages = ('retard_min', 'count')
).round(2).reset_index()

tendance['mois'] = pd.to_datetime(tendance['mois'])
tendance = tendance.sort_values('mois')

print(f"Nombre de mois : {len(tendance)}")
print(tendance)
print(tendance.columns.tolist())


Nombre de mois : 12
         mois  retard_moyen  pct_en_retard  total_passages
0  2025-01-01          2.01          10.62         1721801
1  2025-02-01          2.29          12.83         1328978
2  2025-04-01          2.03          10.67         1541054
3  2025-05-01          2.06          10.90         1678330
4  2025-07-01          1.79           8.68         1654318
5  2025-08-01          1.71           8.10         1597316
6  2025-09-01          2.02          10.41         1766983
7  2025-10-01          2.32          12.90         1819466
8  2025-11-01          2.06          10.75         1535125
9  2025-12-01          1.79           8.65         1797300
10 2026-01-01          1.89           9.44         1638419
11 2026-02-01          1.79           8.67         1650954
['mois', 'retard_moyen', 'pct_en_retard', 'total_passages']


In [37]:

# Graphique
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig3 = make_subplots(specs=[[{'secondary_y': True}]])

fig3.add_trace(go.Bar(
    x=tendance['mois'],
    y=tendance['total_passages'],
    name='Passages',
    marker_color='#d6e4f0',
    opacity=0.6
), secondary_y=True)

fig3.add_trace(go.Scatter(
    x=tendance['mois'],
    y=tendance['pct_en_retard'],
    name='% en retard',
    mode='lines+markers',
    line=dict(color='#bd0026', width=2),
    marker=dict(size=7)
), secondary_y=False)

fig3.add_trace(go.Scatter(
    x=tendance['mois'],
    y=tendance['retard_moyen'],
    name='Retard moyen (min)',
    mode='lines+markers',
    line=dict(color='#fd8d3c', width=2, dash='dot'),
    marker=dict(size=7)
), secondary_y=False)

fig3.update_layout(
    title='Évolution mensuelle de la ponctualité — Réseau Infrabel 2025-2026',
    height=450,
    font_family='Arial',
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_font_size=14,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    xaxis=dict(
        tickformat='%b %Y',
        showgrid=True,
        gridcolor='#f0f0f0'
    )
)
fig3.update_yaxes(title_text='% en retard / Retard moyen (min)', secondary_y=False)
fig3.update_yaxes(title_text='Nombre de passages', secondary_y=True, showgrid=False)

os.makedirs('outputs', exist_ok=True)
fig3.write_html('outputs/tendance_mensuelle.html')
fig3.show()


---
## 8️ Conclusions & Insights

> 

###  Observations clés
- **Gare la plus problématique :** ...
- **Retard moyen global :** ...
- **Tendance :** ...

